<a href="https://colab.research.google.com/github/arctecnologia/Agente-IA-AVDD/blob/main/Agente_MitoIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚽ Agente MitoIA - Protótipo V1
**Desenvolvido por:** Aline da Rocha Cardoso | **Lilicatech AI**

Este notebook contém a prova de conceito (PoC) do Agente MitoIA. Aqui, vamos inicializar o LLM (Google Gemini), configurar o *System Prompt* com a persona de cientista de dados do Cartola FC (Temporada 2026+) e realizar as primeiras requisições de teste para validação de escalação e estratégia.

In [13]:
# ==============================================================================
# PROJETO: AGENTE MITOIA 4.0 - LIVE GROUNDING (GE GATO MESTRE)
# VERSÃO: OFICIAL PREMIUM (SDK 2026) - LILICATECH AI
# OBJETIVO: ELIMINAR ALUCINAÇÕES VIA BUSCA EM TEMPO REAL
# ==============================================================================

# 1. Instalação das dependências atualizadas
!pip install -q -U google-genai gradio

import os
from google import genai
from google.genai import types
from google.colab import userdata
import gradio as gr

# 2. Conexão com o Motor de Busca do Google
try:
    API_KEY = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=API_KEY)
    print("🚀 MitoIA 4.0 Online: Conexão Premium com Google Search Ativada.")
except Exception:
    print("❌ Erro: Configure sua 'GEMINI_API_KEY' nos Secrets do Colab.")

# 3. Configuração da Ferramenta de Busca (Grounding)
TOOLS = [types.Tool(google_search=types.GoogleSearchRetrieval())]

# 4. Núcleo de Inteligência (System Instruction)
MITO_IA_CORE = """
Você é o "MitoIA 4.0 - Squad Architect", a inteligência suprema do Cartola FC 2026.
Sua análise é superior ao Gato Mestre, Chapeleiro.IA e Cartoleiro Neural porque você consome dados em tempo real.

DIRETRIZES CRÍTICAS:
1. GROUNDING OBRIGATÓRIO: Você DEVE acessar o link fornecido pelo usuário para validar os confrontos da rodada. Se o link disser 'Bahia x Cruzeiro', essa é a verdade absoluta.
2. EXTRAÇÃO DE MÉTRICAS: Busque no link os valores de xG (Gols Esperados) e as probabilidades de SG citadas pelos analistas do GE.
3. SQUAD BUILDING: Escale 11 jogadores + Técnico. Use a lógica do Gato Mestre (consistência de scouts) e Chapeleiro (potencial de pontuação individual).
4. CONTEXTO 2026: Interprete as entrelinhas (Lei do Ex, mando de campo, desfalques mencionados no GE).

ESTRUTURA DO RELATÓRIO:
## 📅 VALIDAÇÃO GATO MESTRE (RODADA X)
(Confirmação dos jogos reais encontrados no link)

## 🏟️ ESCALAÇÃO COMPLETA (ESQUEMA TÁTICO)
(Tabela: Posição | Nome | Time | Preço | xP)

## 🧠 JUSTIFICATIVA DATA-DRIVEN
(Análise técnica baseada no xG e SG reais da rodada)

## 🎩 CAPITÃO E VICE
"""

# 5. Função de Processamento
def consultar_e_escalar(rodada, link, perfil, esquema, orcamento):
    user_prompt = f"""
    Acesse o link oficial do Gato Mestre: {link}
    Extraia os dados da Rodada {rodada} do Brasileirão 2026.
    Com base nesses dados REAIS, monte um time para a estratégia '{perfil}' no esquema {esquema} com C$ {orcamento}.
    """

    # Modelos resilientes (tentamos o Flash pela velocidade de busca)
    modelos = ["gemini-1.5-flash", "gemini-3.1-flash-lite", "gemini-flash-latest"]

    for mod in modelos:
        try:
            response = client.models.generate_content(
                model=mod,
                contents=user_prompt,
                tools=TOOLS, # Ativa a busca web
                config=types.GenerateContentConfig(
                    system_instruction=MITO_IA_CORE,
                    temperature=0.0, # Zero para máxima fidelidade aos dados do link
                )
            )
            return response.text
        except Exception as e:
            continue

    return "⚠️ O servidor do Google está sob alta demanda. Tente novamente em 1 minuto."

# 6. Interface Dashboard Lilicatech
with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as ui:
    gr.Markdown(
        """
        # 🏟️ MitoIA 4.0 - Live Grounding Edition
        ### A inteligência que lê o Gato Mestre (GE) em tempo real.
        *Desenvolvido por Aline Cardoso | Lilicatech AI*
        """
    )

    with gr.Row():
        with gr.Column(scale=1):
            link_ge = gr.Textbox(
                label="Link do Gato Mestre (GE)",
                value="https://ge.globo.com/gato-mestre/dicas/noticia/2026/05/07/veja-a-expectativa-de-gols-xg-e-as-dicas-de-sg-para-a-rodada-15-do-cartola-2026.ghtml",
                lines=2
            )
            rodada_num = gr.Number(label="Rodada", value=15)
            perfil_sel = gr.Dropdown(["Agressivo", "Conservador", "Especulador"], label="Estratégia", value="Agressivo")
            esquema_sel = gr.Dropdown(["4-3-3", "3-4-3", "4-4-2", "3-5-2"], label="Tática", value="4-3-3")
            orcamento_val = gr.Slider(80, 320, value=140, label="Patrimônio (C$)")
            btn = gr.Button("🚀 Sincronizar GE e Escalar", variant="primary")

        with gr.Column(scale=2):
            saida_markdown = gr.Markdown(label="Relatório Analítico")

    btn.click(
        fn=consultar_e_escalar,
        inputs=[rodada_num, link_ge, perfil_sel, esquema_sel, orcamento_val],
        outputs=saida_markdown
    )

ui.launch(share=True)

🚀 MitoIA 4.0 Online: Conexão Premium com Google Search Ativada.


/tmp/ipykernel_16905/1973809858.py:80: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue="green")) as ui:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4effb4f039edfd868f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
